# 🐝 Heady Brain — GPU Runtime

**Chat directly with HeadyBrain. Keys load automatically — just run the cells.**

1. Run Cell 1 (loads keys)
2. Run Cell 2 (chat engine with **persistent conversation history**)
3. Run Cell 3 to talk!

### Commands
- `chat("msg")` — continues conversation
- `new_chat()` — fresh conversation
- `save_chat("name")` / `load_chat("name")` — persist across restarts
- `show_history()` — view conversation so far

In [ ]:
# ═══ Cell 1: Load Keys from Heady Memory ═══import base64 as _b, osdef _d(s): return _b.b64decode(s).decode()_vault = {    'gemini': [        (_d('QUl6YVN5RE9MNVlKX042Q0xlVk5ReXRCeURENm5aMEktS3I0WVJr'), 'HC-Heady'),        (_d('QUl6YVN5QllSWEtTckdDVXBfWUxhbGo5R0NRNnRtbjlNZENpVXNn'), 'HC-GCloud'),        (_d('QUl6YVN5Q3hkTlowQUpFVVoxUi1iQWljaDJIa28tRHpSb0w4RjVv'), 'HM-2'),        (_d('QUl6YVN5RENZTXJjYkVONTF1d0lteXR2RUtPeUVFbXVaZGc4TFBr'), 'HM-3'),        (_d('QUl6YVN5QkxUdTBoOVEwOUNyMDVfM19aal8zeWVudDVjTzNpYUhF'), 'HM-4'),        (_d('QUl6YVN5REIxU2MyUHpzUld4TldvaGNJQ0dIQXFRS001WXZSY1hv'), 'HM-Default'),    ],    'openai': [        (_d('c2stcHJvai1mb29fcDRGa2JCWklDTENRcFc4T3VmMlRLTlNZVlBrTnd5Q3lKUlBDQ29xVmFITWVLT3NKSF9Ha0h5bHhKR1JzZW85VHQySVo3QlQzQmxia0ZKT2ZXNThjTTh6LUFDMXY5dUt4YjdkelpoMjNLXy1XM0lmb0U5NkxCME44NXJlVEp2V1Mxa0llOHFxM2ZkTzlsdEVVUHJWOUNWVUE='), 'Seat1'),        (_d('c2stc3ZjYWNjdC0xNzB6NFhLYnI4dE0xTDBRbkpsMEMtS0lOeldaemFLM0E3RVpIaDJMRjNHczJqUUplZEpuallJRUFHb3A2OUwxbGY1WGwyajcyS1QzQmxia0ZKdTNHRnFZUVZxdDVuc2ZxdmlFNm1xUXVyUmlYM19DbXk1TVYxRjlFdUVzSU1odHNZaTVyNmZ2YURnWGRYWWN4VGZ0cUZWZXctd0E='), 'SvcAcct'),    ],    'anthropic': [        (_d('c2stYW50LWFwaTAzLXN4cnZjRkVmdzBZMkJVWnVySVNidmhaS0N0OWg3bTNyRDZXa3hUaDJKekR4U2N0MlFENjZTZTEybjRjbkIxUVM1Nk1IYWNQWFgwVERXUG8wM0ppSU1RLWRRYkJid0FB'), 'Claude-API03'),    ],    'huggingface': [        (_d('aGZfdW1IdXlLV1pVU2tXaG54U3RaQVFORUdtSFNXenBlU0dLaw=='), 'HF-Seat1'),        (_d('aGZfT3lwU0JwSVlNaERrR3BEVkZoYVZJSUxPRmVnUlpWV3hPdA=='), 'HF-Seat2'),    ],    'perplexity': [],}def _get(name):    try:        from google.colab import userdata        v = userdata.get(name)        if v: return v    except: pass    return os.environ.get(name)for name, provider in [('GEMINI_API_KEY','gemini'),('OPENAI_API_KEY','openai'),('ANTHROPIC_API_KEY','anthropic'),('HF_TOKEN','huggingface'),('PERPLEXITY_API_KEY','perplexity')]:    v = _get(name)    if v and not any(k==v for k,_ in _vault.get(provider,[])): _vault[provider].append((v, f'env:{name}'))KEYS = _vaulttotal = sum(len(v) for v in KEYS.values())print(f'🐝 Heady Memory: {total} keys loaded automatically')for p, ks in KEYS.items():    if ks: print(f'  ✅ {p}: {len(ks)} key(s)')    else:  print(f'  ⬜ {p}: none')

In [ ]:
# ═══ Cell 2: Liquid Chat Engine with Persistent History ═══
import json, urllib.request, time, concurrent.futures, os, uuid

SYSTEM = """You are HeadyBrain, the sovereign AI reasoning engine of the Heady ecosystem.
3 accounts (HeadySystems, HeadyConnection, HeadyMe), 5 domains, HuggingFace Spaces, Cloud Run.
Be helpful, precise, and actionable."""

# ─── Persistent Conversation History ───────────────────────────────
conversation_history = []
session_id = str(uuid.uuid4())[:8]
MAX_HISTORY = 50
SESSIONS_DIR = '/content/heady_sessions'

def _trim_history():
    global conversation_history
    if len(conversation_history) > MAX_HISTORY:
        conversation_history = conversation_history[-MAX_HISTORY:]

def new_chat():
    global conversation_history, session_id
    conversation_history = []
    session_id = str(uuid.uuid4())[:8]
    print(f'\U0001f195 New conversation started (session: {session_id})')

def show_history():
    if not conversation_history:
        print('\U0001f4ed No conversation history yet.')
        return
    print(f'\U0001f4dc Conversation ({len(conversation_history)} messages):')
    for i, msg in enumerate(conversation_history):
        role = '\U0001f464 You' if msg['role'] == 'user' else '\U0001f41d Heady'
        preview = msg['content'][:200]
        print(f'  {i+1}. {role}: {preview}')

def save_chat(name='auto'):
    os.makedirs(SESSIONS_DIR, exist_ok=True)
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    data = {'session_id': session_id, 'history': conversation_history, 'saved_at': time.strftime('%Y-%m-%dT%H:%M:%S')}
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'\U0001f4be Saved {len(conversation_history)} messages to {path}')

def load_chat(name='auto'):
    global conversation_history, session_id
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    if not os.path.exists(path):
        print(f'\u274c No saved chat found at {path}')
        return
    with open(path) as f:
        data = json.load(f)
    conversation_history = data.get('history', [])
    session_id = data.get('session_id', name)
    print(f'\U0001f4c2 Loaded {len(conversation_history)} messages (session: {session_id})')


# ─── Provider Calls (each handles its own errors) ───────────────────
def _call_gemini(key, message):
    contents = []
    for msg in conversation_history:
        role = 'model' if msg['role'] == 'assistant' else 'user'
        contents.append({'role': role, 'parts': [{'text': msg['content']}]})
    contents.append({'role': 'user', 'parts': [{'text': message}]})
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={key}'
    data = json.dumps({
        'systemInstruction': {'parts': [{'text': SYSTEM}]},
        'contents': contents,
        'generationConfig': {'maxOutputTokens': 8192}
    }).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
    resp = urllib.request.urlopen(req, timeout=120)
    d = json.loads(resp.read())
    if 'candidates' in d and d['candidates']:
        return d['candidates'][0]['content']['parts'][0]['text']
    raise ValueError(f"No candidates in response: {json.dumps(d)[:200]}")

def _call_openai(key, message):
    messages = [{'role': 'system', 'content': SYSTEM}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': message})
    data = json.dumps({'model': 'gpt-4o', 'messages': messages, 'max_tokens': 8192}).encode()
    req = urllib.request.Request('https://api.openai.com/v1/chat/completions', data=data,
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    d = json.loads(urllib.request.urlopen(req, timeout=120).read())
    if 'choices' in d and d['choices']:
        return d['choices'][0]['message']['content']
    raise ValueError(f"No choices: {json.dumps(d)[:200]}")

def _call_anthropic(key, message):
    messages = list(conversation_history)
    messages.append({'role': 'user', 'content': message})
    data = json.dumps({'model': 'claude-sonnet-4-20250514', 'max_tokens': 8192, 'system': SYSTEM, 'messages': messages}).encode()
    req = urllib.request.Request('https://api.anthropic.com/v1/messages', data=data,
        headers={'Content-Type': 'application/json', 'x-api-key': key, 'anthropic-version': '2023-06-01'})
    d = json.loads(urllib.request.urlopen(req, timeout=120).read())
    if 'content' in d and d['content']:
        return d['content'][0]['text']
    raise ValueError(f"No content: {json.dumps(d)[:200]}")

def _call_perplexity(key, message):
    messages = [{'role': 'system', 'content': SYSTEM}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': message})
    data = json.dumps({'model': 'sonar-pro', 'messages': messages, 'max_tokens': 4096}).encode()
    req = urllib.request.Request('https://api.perplexity.ai/chat/completions', data=data,
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    d = json.loads(urllib.request.urlopen(req, timeout=120).read())
    if 'choices' in d and d['choices']:
        return d['choices'][0]['message']['content']
    raise ValueError(f"No choices: {json.dumps(d)[:200]}")

def _call_huggingface(key, message):
    messages = list(conversation_history)
    messages.append({'role': 'user', 'content': message})
    data = json.dumps({'model': 'deepseek/deepseek-r1-0528', 'messages': messages, 'max_tokens': 4096, 'stream': False}).encode()
    req = urllib.request.Request('https://router.huggingface.co/novita/v3/openai/chat/completions', data=data,
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    d = json.loads(urllib.request.urlopen(req, timeout=120).read())
    if 'choices' in d and d['choices']:
        return d['choices'][0]['message']['content']
    raise ValueError(f"No choices: {json.dumps(d)[:200]}")

PROVIDERS = {
    'gemini': _call_gemini,
    'openai': _call_openai,
    'anthropic': _call_anthropic,
    'huggingface': _call_huggingface,
    'perplexity': _call_perplexity,
}

# Priority order for liquid routing
PROVIDER_ORDER = ['gemini', 'anthropic', 'openai', 'huggingface', 'perplexity']


def _liquid_call(message, preferred=None):
    """Liquid routing: try preferred provider first, then cascade through all others.
    Never crashes. Always returns a response or a clear explanation."""
    errors = []

    # Build the attempt order: preferred first, then all others
    order = list(PROVIDER_ORDER)
    if preferred and preferred in order:
        order.remove(preferred)
        order.insert(0, preferred)

    for provider in order:
        keys = KEYS.get(provider, [])
        if not keys:
            continue
        fn = PROVIDERS.get(provider)
        if not fn:
            continue
        for key, label in keys:
            try:
                s = time.time()
                r = fn(key, message)
                elapsed = int((time.time() - s) * 1000)
                return {'ok': True, 'response': r, 'provider': provider, 'label': label, 'elapsed': elapsed}
            except Exception as e:
                err_msg = str(e)[:120]
                errors.append(f'{provider}/{label}: {err_msg}')
                # Continue to next key / provider

    # All providers failed - return graceful error
    return {
        'ok': False,
        'response': f"All providers exhausted. Errors:\n" + "\n".join(f"  - {e}" for e in errors),
        'provider': 'none',
        'label': 'fallback',
        'elapsed': 0,
        'errors': errors
    }


def chat(message, provider='auto'):
    """Liquid chat: tries your preferred provider, cascades through all others if it fails.
    Conversation history is maintained automatically. Never crashes."""
    if provider == 'all':
        return deep_research(message)

    preferred = None if provider == 'auto' else provider
    result = _liquid_call(message, preferred)

    if result['ok']:
        conversation_history.append({'role': 'user', 'content': message})
        conversation_history.append({'role': 'assistant', 'content': result['response']})
        _trim_history()
        turns = len(conversation_history) // 2
        print(f'\U0001f41d HeadyBrain [{result["provider"]}/{result["label"]}] ({result["elapsed"]}ms, {turns} turns):\n')
        print(result['response'])
        return result['response']
    else:
        print(f'\u274c {result["response"]}')
        return None


def deep_research(message):
    """Query ALL providers simultaneously. Each gets the full conversation history."""
    print('\U0001f52c Deep Research: querying ALL providers...\n')
    tasks = []
    for p in PROVIDER_ORDER:
        keys = KEYS.get(p, [])
        if keys:
            tasks.append((p, keys[0][0], keys[0][1]))

    results = []
    errors = []
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(PROVIDERS[p], k, message): (p, l) for p, k, l in tasks}
        for f in concurrent.futures.as_completed(futs):
            p, l = futs[f]
            try:
                r = f.result(timeout=120)
                print(f'\u2705 {p}/{l}: {len(r)} chars')
                results.append((p, l, r))
            except Exception as e:
                print(f'\u274c {p}/{l}: {str(e)[:80]}')
                errors.append(f'{p}/{l}')

    # Persist best result
    conversation_history.append({'role': 'user', 'content': message})
    if results:
        best = max(results, key=lambda x: len(x[2]))
        conversation_history.append({'role': 'assistant', 'content': best[2]})
    _trim_history()

    total_ms = int((time.time() - start) * 1000)
    print(f'\n=== {len(results)}/{len(tasks)} providers responded ({total_ms}ms) ===\n')
    for p, l, r in results:
        print(f'-- {p.upper()}/{l} ({len(r)} chars) --')
        print(r[:3000])
        if len(r) > 3000:
            print(f'... [{len(r)-3000} more chars]')
        print()
    return results


print(f'\U0001f4ac Liquid Chat ready! Session: {session_id}')
print('  chat("your message")              -> auto-routes (liquid failover)')
print('  chat("msg", "gemini")             -> prefer Gemini, cascade on fail')
print('  chat("msg", "anthropic")          -> prefer Claude, cascade on fail')
print('  chat("msg", "all")               -> ALL providers at once')
print('  new_chat() / show_history() / save_chat("name") / load_chat("name")')


In [ ]:
# ═══ Cell 3: 💬 CHAT ═══
chat("What should we prioritize to get Heady to enterprise grade?")

In [ ]:
# ═══ Cell 4: 🔬 DEEP RESEARCH ═══
deep_research("Enterprise best practices for multi-provider AI with autonomous operations")

In [ ]:
# ═══ Cell 5: 🔄 Interactive Chat (with persistent history) ═══print('🐝 HeadyBrain — type and talk! History is maintained.')print('Prefix: @openai @claude @gemini @hf @all')print('Commands: /new /save [name] /load [name] /history /quit')print()while True:    try:        msg = input('You: ')        if msg.lower() in ('quit', 'exit', 'q', '/quit'):            break        if not msg.strip():            continue        if msg.strip() == '/new':            new_chat()            continue        if msg.strip() == '/history':            show_history()            continue        if msg.strip().startswith('/save'):            parts = msg.strip().split(' ', 1)            name = parts[1] if len(parts) > 1 else 'auto'            save_chat(name)            continue        if msg.strip().startswith('/load'):            parts = msg.strip().split(' ', 1)            name = parts[1] if len(parts) > 1 else 'auto'            load_chat(name)            continue        p = 'auto'        for pfx, prov in [('@openai ', 'openai'), ('@claude ', 'anthropic'), ('@gemini ', 'gemini'), ('@hf ', 'huggingface'), ('@all ', 'all')]:            if msg.startswith(pfx):                p, msg = prov, msg[len(pfx):]                break        print()        chat(msg, p)        print()    except KeyboardInterrupt:        print('🛑 Done.')        break